# 🎙️➡️🎨 From Voice to Vision — Fase 5b: Ottimizzazione robusta + Ensemble

La ricerca precedente premiava configurazioni che convergono in fretta ma generalizzano peggio
(multi-fidelity mismatch). Qui la rendiamo affidabile:
- **Seeding**: gli ottimizzatori partono includendo la configurazione di default (già buona).
- **Spazio di ricerca sensato** (rete non troppo piccola, lr non troppo alto).
- **Retrain dei finalisti** (default + vincitori PSO/FSO/GA) a piena durata → si sceglie il migliore sulla validation.
- **Ensemble a voto soft** dei finalisti → numero finale.

Così il modello finale **non può fare peggio del baseline** e ha una vera chance di superarlo.

In [ ]:
# === 1. Setup + split ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

import numpy as np
from src import config, data_loader, features
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
assert np.array_equal(data["y"], df["emotion_id"].to_numpy())
data["split"] = df["split"].to_numpy()
splits = features.split_arrays(data)
print("Split:", {s: len(splits[s]['y']) for s in ['train','val','test']})

In [ ]:
# === 2. Budget ricerca (con seeding) ===
QUICK = False
if QUICK:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 4, 2, 10
else:
    N_AGENTS, N_ITER, EPOCHS_SEARCH = 5, 4, 18

from src.optimization import pso, fso, ga
from src.optimization.search_space import make_objective, decode, default_u, DEFAULT_HP, DIM
import time
SEED_U = default_u()   # configurazione di default come seme

def run(optimizer, **kw):
    obj = make_objective(splits, epochs=EPOCHS_SEARCH, patience=5)
    t0 = time.time()
    res = optimizer.optimize(obj, DIM, seed=config.SEED, seed_u=SEED_U, **kw)
    res["time_s"] = time.time()-t0
    res["n_evals"] = len(obj.history_evals)
    res["best_hp"] = decode(res["best_u"])
    print(f"→ {res['name']}: robust_val={res['best_fit']:.3f} | eval={res['n_evals']} | {res['time_s']:.0f}s")
    return res

### PSO / FSO / GA (con seeding)

In [ ]:
res_pso = run(pso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_pso['best_hp'])

In [ ]:
res_fso = run(fso, n_particles=N_AGENTS, n_iter=N_ITER); print(res_fso['best_hp'])

In [ ]:
res_ga  = run(ga,  pop_size=N_AGENTS, n_iter=N_ITER); print(res_ga['best_hp'])

## 3) Convergenza

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
allres=[res_pso,res_fso,res_ga]
plt.figure(figsize=(9,5))
for r in allres: plt.plot(range(len(r['history'])), r['history'], marker='o', label=r['name'])
plt.xlabel('iterazione'); plt.ylabel('robust validation accuracy')
plt.title('Convergenza con seeding: PSO vs FSO vs GA'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05b_convergence.png', dpi=120); plt.show()
display(pd.DataFrame([{ 'algoritmo':r['name'],'robust_val':round(r['best_fit'],3),
                       'valutazioni':r['n_evals'],'tempo_s':round(r['time_s']), **r['best_hp']} for r in allres]))

## 4) Retrain dei finalisti a piena durata
Rialleniamo default + vincitori dei tre algoritmi. Scegliamo il migliore sulla **validation**
(niente sbirciate sul test) e teniamo i modelli per l'ensemble.

In [ ]:
from src.models import cnn
finalists = {"default": DEFAULT_HP, "PSO": res_pso['best_hp'],
             "FSO": res_fso['best_hp'], "GA": res_ga['best_hp']}
trained = {}
for name, hp in finalists.items():
    print(f"\n=== Retrain {name}: {hp} ===")
    out = cnn.train_cnn(splits, hp, epochs=60, patience=12, deltas=True, augment=True, verbose=False)
    trained[name] = out
    print(f"  {name}: val={out['best_val_acc']:.3f}  test={out['test_acc']:.3f}")

# migliore singolo scelto sulla VALIDATION
best_name = max(trained, key=lambda k: trained[k]['best_val_acc'])
best = trained[best_name]
print(f"\nMigliore singolo (per val): {best_name}  → test={best['test_acc']:.3f}")

In [ ]:
# === 5) Ensemble a voto soft dei finalisti ===
models = [trained[k]['model'] for k in trained]
ens_acc, ens_probs, y_true = cnn.ensemble_evaluate(models, splits, split_name="test")
print(f"Ensemble (default+PSO+FSO+GA) → test = {ens_acc:.3f}")

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
ens_pred = ens_probs.argmax(1)
print(classification_report(y_true, ens_pred, target_names=config.EMOTIONS, zero_division=0))
plt.figure(figsize=(7,6))
sns.heatmap(confusion_matrix(y_true, ens_pred), annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=config.EMOTIONS, yticklabels=config.EMOTIONS)
plt.title(f'Ensemble — test={ens_acc:.2f}'); plt.xlabel('predetto'); plt.ylabel('reale'); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05b_confusion_ensemble.png', dpi=120); plt.show()

## 6) Progressione finale

In [ ]:
improv = pd.DataFrame([
    {'modello':'SVM (MFCC)','test_acc':0.517},
    {'modello':'CNN baseline (no aug)','test_acc':0.529},
    {'modello':'CNN default (pipeline pot.)','test_acc':round(trained['default']['test_acc'],3)},
    {'modello':f'CNN ottimizzata ({best_name})','test_acc':round(best['test_acc'],3)},
    {'modello':'Ensemble PSO+FSO+GA+def','test_acc':round(ens_acc,3)},
])
plt.figure(figsize=(9,4.5))
sns.barplot(data=improv, x='test_acc', y='modello', color='#4C78A8')
for i,v in enumerate(improv['test_acc']): plt.text(v+0.01,i,f"{v:.2f}",va='center')
plt.xlim(0,1); plt.title("Progressione dell'accuracy sul test (speaker-independent)")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR/'05b_improvement.png', dpi=120); plt.show()
display(improv)

# salvataggi (il best singolo serve alla Fase 6 Grad-CAM)
import json, torch
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR/'best_hp.json','w') as f:
    json.dump({'winner':best_name,'hp':finalists[best_name],
               'val_acc':best['best_val_acc'],'test_acc':best['test_acc'],
               'ensemble_test_acc':ens_acc}, f, indent=2)
with open(config.RESULTS_DIR/'optimization_results.json','w') as f:
    json.dump([{ 'name':r['name'],'robust_val':r['best_fit'],'n_evals':r['n_evals'],
                 'time_s':r['time_s'],'history':r['history'],'hp':r['best_hp']} for r in allres], f, indent=2)
torch.save({'state_dict':best['model'].state_dict(),'hp':finalists[best_name],
            'norm':best['norm'],'in_channels':best['in_channels']}, config.RESULTS_DIR/'best_cnn.pt')
print('✓ salvati best_hp.json, optimization_results.json, best_cnn.pt')

### ✅ Fase 5b completata
Modello finale robusto (≥ baseline) + ensemble come numero di punta.

**Mandami:** tabella PSO/FSO/GA, convergenza, i test dei finalisti, l'accuracy dell'ensemble e la barra finale.
Poi **Fase 6: Explainability** (Grad-CAM, t-SNE, SHAP) — l'altro grande fattore-lode.